# 🛡️ CyberShield AI — v0.1 (Google Colab)

**Objetivo:** detectar padrões suspeitos de falhas de login em logs **fictícios** e explicar o resultado. Esta primeira versão utiliza regras determinísticas, **não Inteligência Artificial**.

Execute as células na ordem. Nenhuma instalação, chave de API ou conexão com sistemas externos é necessária.


In [ ]:
from collections import defaultdict
from datetime import datetime, timedelta, timezone

WINDOW_MINUTES = 10
THRESHOLD = 5

def detect_bruteforce(events, threshold=THRESHOLD, window_minutes=WINDOW_MINUTES):
    """Detecta >= threshold falhas de login por IP numa janela móvel.

    Os resultados indicam suspeita, não confirmam um ataque.
    """
    grouped = defaultdict(list)
    for event in events:
        if event.get("event_type") != "login_failed":
            continue
        ip = event.get("src_ip")
        try:
            timestamp = datetime.fromisoformat(event["timestamp"].replace("Z", "+00:00"))
        except (ValueError, TypeError, KeyError):
            continue
        if not ip or timestamp.tzinfo is None:
            continue
        grouped[ip].append(timestamp.astimezone(timezone.utc))

    alerts = []
    for ip, times in grouped.items():
        times.sort()
        left = 0
        for right, current in enumerate(times):
            while current - times[left] > timedelta(minutes=window_minutes):
                left += 1
            count = right - left + 1
            if count >= threshold:
                alerts.append({
                    "rule_id": "CSAI-001",
                    "source_ip": ip,
                    "failed_attempts": count,
                    "window_minutes": window_minutes,
                    "first_seen": times[left].isoformat(),
                    "last_seen": current.isoformat(),
                    "severity": "high" if count >= threshold * 2 else "medium",
                    "description": "Possível tentativa de força bruta: múltiplas falhas de autenticação.",
                    "recommendation": "Validar com o proprietário do sistema, revisar logs e MFA; não bloquear automaticamente."
                })
                # somente primeiro alerta por IP para evitar notificações duplicadas
                break
    return alerts


## 1. Criar eventos fictícios

Os IPs `192.0.2.0/24`, `198.51.100.0/24` e `203.0.113.0/24` são intervalos reservados para documentação.


In [ ]:
events = [
    {"timestamp": f"2026-11-17T14:0{i}:00Z", "event_type": "login_failed", "src_ip": "192.0.2.10"}
    for i in range(6)
]
events += [
    {"timestamp": "2026-11-17T14:01:00Z", "event_type": "login_success", "src_ip": "198.51.100.20"},
    {"timestamp": "2026-11-17T14:02:00Z", "event_type": "login_failed", "src_ip": "203.0.113.3"},
]
print(f"Eventos carregados: {len(events)}")


## 2. Executar detecção e visualizar alertas

In [ ]:
alerts = detect_bruteforce(events)
if not alerts:
    print("Nenhum alerta nesta amostra.")
else:
    for alert in alerts:
        print(f"⚠️ {alert['rule_id']} | {alert['severity'].upper()} | {alert['source_ip']} | {alert['failed_attempts']} falhas")
        print(f"   {alert['description']}")
        print(f"   Ação sugerida: {alert['recommendation']}")


## 3. Testes mínimos

Valide o comportamento esperado antes de evoluir o detector.

In [ ]:
assert len(alerts) == 1
assert alerts[0]["source_ip"] == "192.0.2.10"
assert detect_bruteforce([{"timestamp": "2026-11-17T14:00:00Z", "event_type": "login_success", "src_ip": "192.0.2.10"}]) == []
assert detect_bruteforce([{"timestamp": f"2026-11-17T{i:02d}:00:00Z", "event_type": "login_failed", "src_ip": "192.0.2.10"} for i in range(5)]) == []
print("✅ Testes concluídos!")


## Próximos passos

1. Adicionar análise de CSV com validação de campos e formatos.
2. Documentar limites, falsos positivos e privacidade.
3. Integrar explicações assistidas por IA **somente após** estabelecer critérios de avaliação e consentimento para dados enviados.
